Markov Decision Process

In [ ]:
import numpy as np

# Environment Setup
wall = [(1, 1)]  # List of wall positions
terminal_states = [(1, 3), (2, 3)]  # Terminal states
actions = ['L', 'R', 'U', 'D']  # Possible actions (Left, Right, Up, Down)

# Non-deterministic action probabilities (for stochastic actions)
action_probability = {'L': 0.25, 'R': 0.25, 'U': 0.25, 'D': 0.25}

# Stochastic effects due to perpendicular actions
environment_left = {'L': 'D', 'R': 'U', 'U': 'L', 'D': 'R'}
environment_right = {'L': 'U', 'R': 'D', 'U': 'R', 'D': 'L'}

# Grid dimensions (3x4 grid)
rows, cols = 3, 4

# Check if the state is valid (not a wall or out of bounds)
def is_valid(i, j):
    return (i, j) not in wall and 0 <= i < rows and 0 <= j < cols

# Transition function based on action
def transition(action, i, j):
    if action == 'L':  # Move Left
        return (i, j - 1)
    elif action == 'R':  # Move Right
        return (i, j + 1)
    elif action == 'U':  # Move Up
        return (i + 1, j)
    elif action == 'D':  # Move Down
        return (i - 1, j)

# Compute the value for a given state (i, j) under the optimal policy
def value_function(i, j, reward_matrix, V, gamma=1.0):
    value = 0
    for action in actions:
        # Intended action (probability 0.8)
        state_x, state_y = transition(action, i, j)
        if is_valid(state_x, state_y):
            desired_value = reward_matrix[state_x][state_y] + gamma * V[state_x][state_y]
        else:
            desired_value = reward_matrix[i][j] + gamma * V[i][j]

        # Left stochastic action (probability 0.1)
        state_x, state_y = transition(environment_left[action], i, j)
        if is_valid(state_x, state_y):
            left_value = reward_matrix[state_x][state_y] + gamma * V[state_x][state_y]
        else:
            left_value = reward_matrix[i][j] + gamma * V[i][j]

        # Right stochastic action (probability 0.1)
        state_x, state_y = transition(environment_right[action], i, j)
        if is_valid(state_x, state_y):
            right_value = reward_matrix[state_x][state_y] + gamma * V[state_x][state_y]
        else:
            right_value = reward_matrix[i][j] + gamma * V[i][j]

        # Weighted sum of values
        action_value = 0.8 * desired_value + 0.1 * left_value + 0.1 * right_value
        value += action_value * action_probability[action]

    return value

# Value iteration function to compute optimal value function
def value_iteration(reward, epsilon=1e-8, gamma=1.0, max_iterations=200):
    V = np.zeros((rows, cols))  # Initialize value function V(s) to zero
    reward_matrix = np.full((rows, cols), reward)  # Initialize reward matrix

    # Set terminal state rewards
    reward_matrix[2][3] = 1  # Goal state (1 point)
    reward_matrix[1][3] = -1  # Penalty state (-1 point)

    # Initialization: Start with an initial guess for the value function (typically zeros)
    iteration = 0
    while iteration < max_iterations:
        delta = 0
        V_new = np.copy(V)

        # Value Updates: For each state, update its value based on the expected rewards
        # of taking each action, considering the stochastic nature of the environment.
        for i in range(rows):
            for j in range(cols):
                if (i, j) in terminal_states or (i, j) in wall:
                    continue  # Skip terminal and wall states

                # Compute the new value for state (i, j)
                V_new[i][j] = value_function(i, j, reward_matrix, V, gamma)
                delta = max(delta, abs(V_new[i][j] - V[i][j]))

        V = np.copy(V_new)
        iteration += 1

        # If the value function has converged (delta is small enough), stop iterating
        if delta < epsilon:
            print(f"Converged after {iteration} iterations.")
            break

    # Extract the Optimal Policy: After convergence, the optimal policy can be derived
    # by choosing the action that maximizes the expected value from each state.
    policy = np.full((rows, cols), '')
    for i in range(rows):
        for j in range(cols):
            if (i, j) in terminal_states or (i, j) in wall:
                continue  # Skip terminal and wall states

            best_action_value = -float('inf')
            best_action = None
            for action in actions:
                state_x, state_y = transition(action, i, j)
                if is_valid(state_x, state_y):
                    desired_value = reward_matrix[state_x][state_y] + gamma * V[state_x][state_y]
                else:
                    desired_value = reward_matrix[i][j] + gamma * V[i][j]

                state_x, state_y = transition(environment_left[action], i, j)
                if is_valid(state_x, state_y):
                    left_value = reward_matrix[state_x][state_y] + gamma * V[state_x][state_y]
                else:
                    left_value = reward_matrix[i][j] + gamma * V[i][j]

                state_x, state_y = transition(environment_right[action], i, j)
                if is_valid(state_x, state_y):
                    right_value = reward_matrix[state_x][state_y] + gamma * V[state_x][state_y]
                else:
                    right_value = reward_matrix[i][j] + gamma * V[i][j]

                action_value = 0.8 * desired_value + 0.1 * left_value + 0.1 * right_value
                total_value = action_value * action_probability[action]

                # Update the best action based on the highest value
                if total_value > best_action_value:
                    best_action_value = total_value
                    best_action = action

            policy[i][j] = best_action

    return V, policy

In [ ]:
# Main execution to compute value functions and optimal policies for different rewards
rewards = [-0.04, -2, 0.1, 0.02, 1]  # Different reward settings
print("Value Functions and Optimal Policies corresponding to optimal policy\n")
for reward in rewards:
    print(f"For r(S) : {reward}")
    V_optimal, optimal_policy = value_iteration(reward)

    # Print the value function after convergence
    print("Value Function:")
    for i in range(rows-1, -1, -1):  # Print from bottom to top of the grid
        for j in range(cols):
            print(f"{V_optimal[i][j]:.2f}", end=" | ")
        print("")

    # Print the optimal policy after convergence
    print("Optimal Policy:")
    for i in range(rows-1, -1, -1):
        for j in range(cols):
            print(f"{optimal_policy[i][j]}", end=" | ")
        print("")

    print("\n")

Value Functions and Optimal Policies corresponding to optimal policy

For r(S) : -0.04
Value Function:
-1.23 | -0.83 | -0.28 | 0.00 | 
-1.47 | 0.00 | -0.87 | 0.00 | 
-1.55 | -1.46 | -1.22 | -1.17 | 
Optimal Policy:
R | R | R |  | 
U |  | U |  | 
R | R | U | U | 


For r(S) : -2
Value Function:
-59.68 | -45.99 | -24.31 | 0.00 | 
-65.37 | 0.00 | -21.93 | 0.00 | 
-63.07 | -52.77 | -34.48 | -20.74 | 
Optimal Policy:
R | R | R |  | 
U |  | R |  | 
R | R | R | U | 


For r(S) : 0.1
Value Function:
2.94 | 2.39 | 1.44 | 0.00 | 
3.10 | 0.00 | 0.63 | 0.00 | 
2.85 | 2.20 | 1.15 | 0.23 | 
Optimal Policy:
D | L | L |  | 
L |  | U |  | 
U | L | L | L | 


For r(S) : 0.02
Value Function:
0.56 | 0.55 | 0.46 | 0.00 | 
0.49 | 0.00 | -0.23 | 0.00 | 
0.34 | 0.11 | -0.20 | -0.57 | 
Optimal Policy:
U | L | R |  | 
U |  | U |  | 
U | L | L | L | 


For r(S) : 1
Value Function:
29.78 | 23.13 | 12.48 | 0.00 | 
32.44 | 0.00 | 10.30 | 0.00 | 
31.10 | 25.76 | 16.42 | 9.21 | 
Optimal Policy:
D | L | L |  | 
L |  |